In [4]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# CONFIG
# ============================================================

DATASETS = {
    "original": Path("./../MIMICEL_data/mimicel.csv"),
    "train": Path("./../MIMICEL_data/mimicel_train.csv"),
    "val": Path("./../MIMICEL_data/mimicel_val.csv"),
    "test": Path("./../MIMICEL_data/mimicel_test.csv"),
}

OUT_DIR = Path("./results/compare")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CASE_COL = "stay_id"
TIME_COL = "timestamps"
ACT_COL = "activity"

TOP_N_ACTIVITY = 10
TOP_N_DFG = 15
BINS = 80


# ============================================================
# LOAD
# ============================================================

def load_data(path: Path) -> pd.DataFrame:
    df = pd.read_csv(
        path,
        usecols=[CASE_COL, TIME_COL, ACT_COL],
        low_memory=False
    )

    df[CASE_COL] = df[CASE_COL].astype(str)
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")

    df = df.dropna(
        subset=[CASE_COL, TIME_COL, ACT_COL]
    )

    df = df.sort_values(
        [CASE_COL, TIME_COL]
    ).reset_index(drop=True)

    return df


print("[1] Loading datasets...")

dfs = {}

for name, path in DATASETS.items():
    print(f"  - {name}: {path}")
    dfs[name] = load_data(path)
    print(
        f"    events={len(dfs[name]):,}, "
        f"cases={dfs[name][CASE_COL].nunique():,}, "
        f"activities={dfs[name][ACT_COL].nunique():,}"
    )


# ============================================================
# FEATURE EXTRACTION
# ============================================================

def activity_ratio(df: pd.DataFrame) -> pd.Series:
    return df[ACT_COL].value_counts(normalize=True)


def trace_length(df: pd.DataFrame) -> pd.Series:
    return df.groupby(CASE_COL).size()


def case_duration_hour(df: pd.DataFrame) -> pd.Series:
    duration = (
        df.groupby(CASE_COL)[TIME_COL]
        .agg(["min", "max"])
    )

    return (
        duration["max"] - duration["min"]
    ).dt.total_seconds() / 3600


def inter_event_minutes(df: pd.DataFrame) -> pd.Series:
    tmp = df.copy()

    tmp["prev_time"] = (
        tmp.groupby(CASE_COL)[TIME_COL]
        .shift(1)
    )

    out = (
        tmp[TIME_COL] - tmp["prev_time"]
    ).dt.total_seconds() / 60

    out = out.dropna()
    out = out[out >= 0]

    return out


def dfg_ratio(df: pd.DataFrame) -> pd.Series:
    tmp = df.copy()

    tmp["next_activity"] = (
        tmp.groupby(CASE_COL)[ACT_COL]
        .shift(-1)
    )

    dfg = tmp.dropna(
        subset=["next_activity"]
    ).copy()

    dfg["transition"] = (
        dfg[ACT_COL].astype(str)
        + " -> "
        + dfg["next_activity"].astype(str)
    )

    return dfg["transition"].value_counts(normalize=True)


def start_activity_ratio(df: pd.DataFrame) -> pd.Series:
    first = df.groupby(CASE_COL)[ACT_COL].first()
    return first.value_counts(normalize=True)


def end_activity_ratio(df: pd.DataFrame) -> pd.Series:
    last = df.groupby(CASE_COL)[ACT_COL].last()
    return last.value_counts(normalize=True)


# ============================================================
# PLOT HELPERS
# ============================================================

def annotate_grouped_bars(
    ax,
    as_percent=True,
    fontsize=7,
    rotation=90
):
    for container in ax.containers:
        for bar in container:
            height = bar.get_height()

            if np.isnan(height) or height <= 0:
                continue

            if as_percent:
                label = f"{height * 100:.2f}%"
            else:
                label = f"{height:.3f}"

            ax.text(
                bar.get_x() + bar.get_width() / 2,
                height,
                label,
                ha="center",
                va="bottom",
                fontsize=fontsize,
                rotation=rotation
            )


def grouped_bar_plot(
    distributions: dict,
    keys,
    title: str,
    ylabel: str,
    out_path: Path,
    as_percent=True
):
    labels = list(distributions.keys())

    keys = list(keys)
    x = np.arange(len(keys))
    width = 0.18

    fig, ax = plt.subplots(figsize=(16, 7))

    for i, label in enumerate(labels):
        values = (
            distributions[label]
            .reindex(keys, fill_value=0)
            .values
        )

        ax.bar(
            x + (i - 1.5) * width,
            values,
            width,
            label=label
        )

    annotate_grouped_bars(
        ax,
        as_percent=as_percent,
        fontsize=7,
        rotation=90
    )

    ax.set_xticks(x)
    ax.set_xticklabels(
        keys,
        rotation=45,
        ha="right"
    )

    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.legend()

    # annotation 공간 확보
    ymax = ax.get_ylim()[1]
    ax.set_ylim(0, ymax * 1.18)

    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.close()


def overlay_hist_plot(
    series_dict: dict,
    title: str,
    xlabel: str,
    out_path: Path,
    bins=BINS,
    log_y=False,
    upper_quantile=0.99
):
    valid_series = [
        s.dropna()
        for s in series_dict.values()
        if len(s.dropna()) > 0
    ]

    cutoff = max(
        s.quantile(upper_quantile)
        for s in valid_series
    )

    plt.figure(figsize=(10, 5))

    for label, series in series_dict.items():
        s = series.dropna()
        s = s[s <= cutoff]

        plt.hist(
            s,
            bins=bins,
            alpha=0.35,
            density=True,
            label=label
        )

    if log_y:
        plt.yscale("log")

    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Density")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.close()


def save_distribution_table(
    distributions: dict,
    keys,
    out_path: Path
):
    result = pd.DataFrame(index=list(keys))

    for name, dist in distributions.items():
        result[name] = dist.reindex(keys, fill_value=0)

    result.to_csv(out_path, encoding="utf-8-sig")


# ============================================================
# 1. ACTIVITY DISTRIBUTION
# ============================================================

print("[2] Activity distribution comparison...")

activity_dists = {
    name: activity_ratio(df)
    for name, df in dfs.items()
}

top_activities = (
    activity_dists["original"]
    .sort_values(ascending=False)
    .head(TOP_N_ACTIVITY)
    .index
)

grouped_bar_plot(
    distributions=activity_dists,
    keys=top_activities,
    title="Activity Distribution Comparison",
    ylabel="Ratio",
    out_path=OUT_DIR / "activity_distribution_comparison.png",
    as_percent=True
)

save_distribution_table(
    activity_dists,
    top_activities,
    OUT_DIR / "activity_distribution_comparison.csv"
)


# ============================================================
# 2. TRACE LENGTH DISTRIBUTION
# ============================================================

print("[3] Trace length distribution comparison...")

trace_lengths = {
    name: trace_length(df)
    for name, df in dfs.items()
}

overlay_hist_plot(
    series_dict=trace_lengths,
    title="Trace Length Distribution Comparison",
    xlabel="Number of Events per Case",
    out_path=OUT_DIR / "trace_length_distribution_comparison.png"
)

overlay_hist_plot(
    series_dict=trace_lengths,
    title="Trace Length Distribution Comparison - Log Scale",
    xlabel="Number of Events per Case",
    out_path=OUT_DIR / "trace_length_distribution_comparison_log.png",
    log_y=True
)


# ============================================================
# 3. CASE DURATION DISTRIBUTION
# ============================================================

print("[4] Case duration distribution comparison...")

case_durations = {
    name: case_duration_hour(df)
    for name, df in dfs.items()
}

overlay_hist_plot(
    series_dict=case_durations,
    title="Case Duration Distribution Comparison",
    xlabel="Duration Hours",
    out_path=OUT_DIR / "case_duration_distribution_comparison.png"
)

overlay_hist_plot(
    series_dict=case_durations,
    title="Case Duration Distribution Comparison - Log Scale",
    xlabel="Duration Hours",
    out_path=OUT_DIR / "case_duration_distribution_comparison_log.png",
    log_y=True
)


# ============================================================
# 4. INTER-EVENT TIME DISTRIBUTION
# ============================================================

print("[5] Inter-event time distribution comparison...")

inter_events = {
    name: inter_event_minutes(df)
    for name, df in dfs.items()
}

overlay_hist_plot(
    series_dict=inter_events,
    title="Inter-Event Time Distribution Comparison",
    xlabel="Minutes Between Events",
    out_path=OUT_DIR / "inter_event_time_distribution_comparison.png"
)

overlay_hist_plot(
    series_dict=inter_events,
    title="Inter-Event Time Distribution Comparison - Log Scale",
    xlabel="Minutes Between Events",
    out_path=OUT_DIR / "inter_event_time_distribution_comparison_log.png",
    log_y=True
)


# ============================================================
# 5. DFG TRANSITION DISTRIBUTION
# ============================================================

print("[6] DFG transition distribution comparison...")

dfg_dists = {
    name: dfg_ratio(df)
    for name, df in dfs.items()
}

top_dfg = (
    dfg_dists["original"]
    .sort_values(ascending=False)
    .head(TOP_N_DFG)
    .index
)

grouped_bar_plot(
    distributions=dfg_dists,
    keys=top_dfg,
    title="DFG Transition Distribution Comparison",
    ylabel="Ratio",
    out_path=OUT_DIR / "dfg_transition_distribution_comparison.png",
    as_percent=True
)

save_distribution_table(
    dfg_dists,
    top_dfg,
    OUT_DIR / "dfg_transition_distribution_comparison.csv"
)


# ============================================================
# 6. START ACTIVITY DISTRIBUTION
# ============================================================

print("[7] Start activity distribution comparison...")

start_dists = {
    name: start_activity_ratio(df)
    for name, df in dfs.items()
}

top_start = (
    start_dists["original"]
    .sort_values(ascending=False)
    .index
)

grouped_bar_plot(
    distributions=start_dists,
    keys=top_start,
    title="Trace Start Activity Distribution Comparison",
    ylabel="Ratio",
    out_path=OUT_DIR / "trace_start_activity_distribution_comparison.png",
    as_percent=True
)

save_distribution_table(
    start_dists,
    top_start,
    OUT_DIR / "trace_start_activity_distribution_comparison.csv"
)


# ============================================================
# 7. END ACTIVITY DISTRIBUTION
# ============================================================

print("[8] End activity distribution comparison...")

end_dists = {
    name: end_activity_ratio(df)
    for name, df in dfs.items()
}

top_end = (
    end_dists["original"]
    .sort_values(ascending=False)
    .index
)

grouped_bar_plot(
    distributions=end_dists,
    keys=top_end,
    title="Trace End Activity Distribution Comparison",
    ylabel="Ratio",
    out_path=OUT_DIR / "trace_end_activity_distribution_comparison.png",
    as_percent=True
)

save_distribution_table(
    end_dists,
    top_end,
    OUT_DIR / "trace_end_activity_distribution_comparison.csv"
)


# ============================================================
# 8. SUMMARY TABLE
# ============================================================

print("[9] Writing summary table...")

summary = []

for name, df in dfs.items():
    tl = trace_lengths[name]
    cd = case_durations[name]
    ie = inter_events[name]

    summary.append({
        "split": name,
        "n_cases": df[CASE_COL].nunique(),
        "n_events": len(df),
        "n_activities": df[ACT_COL].nunique(),

        "mean_trace_length": tl.mean(),
        "median_trace_length": tl.median(),
        "max_trace_length": tl.max(),

        "mean_case_duration_hour": cd.mean(),
        "median_case_duration_hour": cd.median(),
        "max_case_duration_hour": cd.max(),

        "mean_inter_event_minutes": ie.mean(),
        "median_inter_event_minutes": ie.median(),
        "max_inter_event_minutes": ie.max(),
    })

summary_df = pd.DataFrame(summary)
summary_df.to_csv(
    OUT_DIR / "split_summary.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# DONE
# ============================================================

print("Done.")
print("Saved to:", OUT_DIR.resolve())

[1] Loading datasets...
  - original: ..\MIMICEL_data\mimicel.csv
    events=7,568,824, cases=425,028, activities=6
  - train: ..\MIMICEL_data\mimicel_train.csv
    events=55,725, cases=3,150, activities=6
  - val: ..\MIMICEL_data\mimicel_val.csv
    events=7,947, cases=450, activities=6
  - test: ..\MIMICEL_data\mimicel_test.csv
    events=16,104, cases=900, activities=6
[2] Activity distribution comparison...
[3] Trace length distribution comparison...
[4] Case duration distribution comparison...
[5] Inter-event time distribution comparison...
[6] DFG transition distribution comparison...
[7] Start activity distribution comparison...
[8] End activity distribution comparison...
[9] Writing summary table...
Done.
Saved to: D:\University\4-1\6Process_Mining\PM_Assessment_Logs\02_Data_EDA\results\compare
